In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import pandas as pd
import re
from fuzzywuzzy import fuzz
import numpy as np

pd.set_option('display.max_rows', None)

In [2]:
# url = "https://www.vegasinsider.com/college-football/odds/las-vegas/"
# soup = BeautifulSoup(requests.get(url).content, "html.parser")

# # clean-up the cells:
# for br in soup.select("br"):
#     br.replace_with("\n")

# df = pd.read_html(str(soup.select_one(".frodds-data-tbl")))[0]

# # set column names:
# # df.columns = ['col1', 'col2', ...]

# df.to_csv("data.csv", index=False)
# print(df)

In [3]:
url = "https://www.vegasinsider.com/college-football/odds/las-vegas/"
tab = pd.read_html(url)

In [4]:
# might have to adjust these weekly based on what the available sources are on the website
# can always capture it from the `cleaned` dataframe below, before dropping columns
# it'll probably error out if you have the wrong columns anyway
sources = [
    'Bet365',
    'BetMGM',
    'DraftKings',
    'HardRock',
    'Caesars',
    # 'ESPNBet',
    'FanDuel',
    'Fanatics',
    # 'BallyBet',
    # 'RiversCasino',
    'Consensus'
          ]

In [5]:
# this cell sets up a breakpoint if the vegas odds reporting website has a game that has already finished
keeps = []
for i, row in enumerate(tab[0]['Time']):
    if type(row) != str:
        continue
    if row == 'Final':
        print('Hit a game that already finished, exiting loop')
        break
    try: 
        check = int(row.split()[0])
        keeps.append(i)
    except:
        continue

intermediate = tab[0].iloc[keeps]

Hit a game that already finished, exiting loop


In [6]:
cleaned = pd.DataFrame(columns=intermediate.columns)
for row in intermediate.iterrows():
    kill = False
    odds = []
    skip = False
    for source in sources:
        check = type(row[1][source]) == str
        skip += check
        odds.append(row[1][source])
    skip = not bool(skip)
    if skip:
        continue
    skip = True
    for odd in odds:
        if type(odd) != str:
            continue
        if odd[0] == 'o':
            kill = True
    if kill:
        break
    cleaned.loc[len(cleaned)] = row[1]
    
# cleaned = cleaned.drop('Unnamed: 12', axis=1)
# cleaned = cleaned.drop('Unnamed: 11', axis=1)
# cleaned = cleaned.drop('Unnamed: 8', axis=1)
cleaned = cleaned.drop(list(cleaned.filter(regex='Unnamed')), axis=1)
cleaned = cleaned.drop('Open', axis=1)
    
for i, row in cleaned.iterrows():
    for source in sources:
        if type(row[source]) != str:
            continue
        if row[source].split()[0][0] == '+':
            num = row[source].split()[0][1:]
        else:
            num = row[source].split()[0]
        if num == 'PK':
            num = 0.
        else:
            num = float(num)
        cleaned.loc[i, source] = float(num)
        
for i, row in cleaned.iterrows():
    result = ''.join([i for i in row['Time'] if not i.isdigit()])
    result = result[1:]
    cleaned.loc[i, 'Time'] = result
cleaned = cleaned.rename(columns={'Time': 'Team'})
cleaned['index'] = np.arange(len(cleaned))

ValueError: could not convert string to float: '--6.5'

In [ ]:
cleaned

In [ ]:
url2 = 'https://www.officefootballpool.com/analysis.cfm?p=16'
# https://www.officefootballpool.com/analysis.cfm?p=16
# url2 = 'https://www.officefootballpool.com/admin.cfm?p=42&pill=2&weekid=634'
# tab2 = pd.read_html(url2)

In [ ]:
# the number for weekid needs to be incremented each week
# you probably just have to use trial/error to see what the soup.find_all in the
# next cell finds to determine what the starting week is/what weekid corresponds to the week you're trying to get
data = {'year': '2025',
        'weekid': 651,
       'sportid': 'FBS'}
soup = BeautifulSoup(requests.post(url2, data=data).content, "html.parser")

In [ ]:
print(soup.find_all('span')[0].text)
print(soup.find_all('span')[1].text)
print(soup.find_all('span')[2].text)
print(soup.find_all('span')[3].text)
print(soup.find_all('span')[4].text)
print(soup.find_all('span')[5].text)
print(soup.find_all('span')[6].text)
print(soup.find_all('span')[7].text)
print(soup.find_all('span')[8].text)
print(soup.find_all('span')[9].text)
print(soup.find_all('span')[10].text)
print(soup.find_all('span')[11].text)
print(soup.find_all('span')[12].text)

In [ ]:
for i in range(50):
    print(soup.find_all('span')[i].text)

In [ ]:
ofp = pd.DataFrame(columns=['Team', 'Spread'])

# always lop off the last three lines (though I don't remember exactly why
for i, span in enumerate(soup.find_all('span')[:-3]):
# sometimes might have to do something like the next line to skip over games that have already happened
# games that have already happened don't get parsed correctly
# for i, span in enumerate(soup.find_all('span')[6:-3]):
    capture = re.split(r'[()]', span.text)
    team = re.split(r'[()]', span.text)[0].strip()
    ofp.loc[i, 'Team'] = team
    if i == 0:
        continue
    if i % 2 == 0:
        continue
    split = float(re.split(r'[()]', span.text)[-1].strip())
    ofp.loc[i, 'Spread'] = split
    ofp.loc[i-1, 'Spread'] = -1*split
    
ofp = ofp[ofp['Spread'] != 0.0]
ofp = ofp.reset_index(drop=True)

In [ ]:
# a couple of hard coded conversions that seemed to always get messed up and not resolved with fuzzywuzzy
conversions = {
    'S. Florida': 'South Florida',
    'UL Lafayette': 'Louisiana',
              }
for key in conversions.keys():
    print(key)
    idx = np.where(ofp['Team'] == key)[0]
    print(idx)
    ofp.loc[idx, 'Team'] = conversions[key]

In [ ]:
# matching team names across vegas odds and OFP
ratios = []
matches = []
for team in ofp['Team']:
    max_ratio = 0
    matched = ''
    for anchor in cleaned['Team']:
        # ratio = fuzz.token_set_ratio(team, anchor)
        ratio = fuzz.ratio(team, anchor)
        if ratio > max_ratio:
            max_ratio = ratio
            matched = anchor
    ratios.append(max_ratio)
    matches.append(matched)

In [110]:
ofp['matched'] = matches 
ofp['ratios'] = ratios

In [111]:
for row in ofp.iterrows():
    if row[1].matched == 'UCF':
        print(row[1].Team, ':', row[1].matched, ':', row[1].ratios)

In [112]:
final = pd.merge(cleaned, ofp, how='left', left_on='Team', right_on='matched')

In [113]:
name, count = np.unique(final['Team_x'], return_counts=True)

In [114]:
doubles = name[np.where(count > 1)[0]]
for double in doubles:
    print(double, '    was matched twice. Will resolve with ratios')

BYU     was matched twice. Will resolve with ratios


In [115]:
final = final.sort_values('ratios', ascending=False).drop_duplicates('Team_x').sort_index()
final = final.reset_index(drop=True)

In [116]:
for double in doubles:
    idx = np.where(final['Team_x'] == double)[0]
    print(f'{double} resolved as {final.iloc[idx]['matched'].values}')
    # print(f'{double} resolved as {final.iloc[idx]['matched'].values}')
    # print('blah')

BYU resolved as ['BYU']


In [117]:
# final = final.drop(columns=['HardRock', 'Fanatics'])

In [118]:
final[:1]

,Team_x,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,index,Team_y,Spread,matched,ratios
0,Toledo,11.0,10.5,11.5,11.0,11.5,10.5,11.0,+11 -114 +,11.5,0,Toledo,10.5,Toledo,100.0


In [119]:
# final['Spread_ave'] = final.iloc[:, [1,2,3,4,5,6,7,8]].mean(axis=1)
# final['Spread_ave'] = final.iloc[:, [1,2,3,4,5,6,7,8]].mean(axis=1)
final['Spread_ave'] = final[sources].mean(axis=1)
final['Spread_diff_ave'] = final['Spread_ave'] - final['Spread']
final['Spread_diff_consensus'] = final['Consensus'] - final['Spread']

In [120]:
final.sort_values(by='Spread_diff_ave')

,Team_x,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus,index,Team_y,Spread,matched,ratios,Spread_ave,Spread_diff_ave,Spread_diff_consensus
3,Southern Miss,-1.5,-1.5,-1.5,-1.5,-1.5,-1.5,-1.0,+1 -114 +,-1.5,3,Southern Miss,-0.5,Southern Miss,100.0,-1.4375,-0.9375,-1.0
1,Louisville,-11.0,-10.5,-11.5,-11.0,-11.5,-10.5,-11.0,-11 -108 +,-11.5,1,Louisville,-10.5,Louisville,100.0,-11.0625,-0.5625,-1.0
16,Penn State,3.0,3.0,3.0,3.0,3.5,3.5,3.0,+3.5 -113 +,3.0,16,Penn St.,3.5,Penn State,78.0,3.125,-0.375,-0.5
13,UTSA,-6.0,-6.0,-6.0,-5.5,-5.5,-6.5,-5.5,-6.5 -112 +,-6.0,13,UTSA,-5.5,UTSA,100.0,-5.875,-0.375,-0.5
28,LSU,2.5,3.0,3.0,3.0,2.5,2.5,3.0,+3 -117 +,3.0,28,LSU,3.0,LSU,100.0,2.8125,-0.1875,0.0
31,Appalachian State,7.0,7.5,7.0,7.5,7.5,7.5,7.5,+7.5 -115 +,7.0,31,Appalachian State,7.5,Appalachian State,100.0,7.3125,-0.1875,-0.5
15,East Carolina,10.0,10.0,10.0,10.0,9.5,9.5,10.0,+9.5 -109 +,10.0,15,East Carolina,10.0,East Carolina,100.0,9.875,-0.125,0.0
6,California,1.0,1.5,1.5,1.5,1.5,1.5,1.5,+1.5 -109 +,1.5,6,California,1.5,California,100.0,1.4375,-0.0625,0.0
27,Missouri,-4.0,-4.5,-4.0,-4.0,-4.5,-3.5,-4.0,-4 -112 +,-4.0,27,Missouri,-4.0,Missouri,100.0,-4.0625,-0.0625,0.0
8,Central Michigan,10.5,10.5,10.5,10.5,10.5,9.5,11.0,+10.5 -112 +,10.5,8,Central Mich,10.5,Central Michigan,86.0,10.4375,-0.0625,0.0


# NFL

In [7]:
url3 = "https://www.vegasinsider.com/nfl/odds/las-vegas/"
tab = pd.read_html(url3)

In [8]:
sources = [
    'Bet365',
    'BetMGM',
    'DraftKings',
    'Caesars',
    # 'ESPNBet',
    'FanDuel',
    'HardRock',
    'Fanatics',
    'RiversCasino',
    'Consensus'
          ]

In [9]:
# sources = [
    # 'Bet365',
    # 'BetMGM',
    # 'DraftKings',
    # 'Caesars',
    # 'ESPNBet',
    # 'FanDuel',
    # 'HardRock',
    # 'Fanatics',
    # 'RiversCasino',
    # 'Consensus'
          # ]

In [10]:
keeps = []
for i, row in enumerate(tab[0]['Time']):
    if type(row) != str:
        continue
    if row == 'Final':
        print('Hit a game that already finished, exiting loop')
        break
    try: 
        check = int(row.split()[0])
        keeps.append(i)
    except:
        continue

intermediate = tab[0].iloc[keeps]

In [13]:
cleaned

,Time,Bet365,BetMGM,DraftKings,Caesars,FanDuel,HardRock,Fanatics,RiversCasino,Consensus
0,451 Patriots,3.5,3.5,3.5,3.5,3.5,--4.5 -105 +,+3.5 -115 +,+3.5 -109 +,+3.5 -115
1,452 Seahawks,-3.5 -110 +,-3.5 -110 +,-3.5 -105 +,-3.5 -110 +,-3.5 -112 +,+-4.5 -110 +,-3.5 -105 +,-3.5 -110 +,-3.5 -105
2,453 49ers,+3.5 -105 +,+3.5 -110 +,+3.5 -108 +,+3.5 -110 +,+3.5 -105 +,+3.5 -105 +,+3.5 -110 +,+3.5 -110 +,+3.5 -108
3,454 Rams,-3.5 -115 +,-3.5 -110 +,-3.5 -112 +,-3.5 -109 +,-3.5 -115 +,-3.5 -115 +,-3.5 -110 +,-3.5 -109 +,-3.5 -112
4,455 Browns,+7.5 -110 +,+7.5 -110 +,+7.5 -110 +,+7.5 -107 +,+7.5 -105 +,+8.5 -115 +,+8.5 -110 +,+7.5 -110 +,+7.5 -110
5,456 Jaguars,-7.5 -110 +,-7.5 -110 +,-7.5 -110 +,-7.5 -113 +,-7.5 -115 +,-8.5 -105 +,-8.5 -110 +,-7.5 -108 +,-7.5 -110
6,457 Buccaneers,+3.5 -105 +,+3.5 -110 +,+3.5 -105 +,+3.5 -108 +,+3.5 -110 +,-3.5 -105 +,+4 -110 +,+3.5 -108 +,+3.5 -105
7,458 Bengals,-3.5 -115 +,-3.5 -110 +,-3.5 -115 +,-3.5 -112 +,-3.5 -110 +,+3.5 -115 +,-4 -110 +,-3.5 -113 +,-3.5 -115
8,459 Ravens,-3.5 -105 +,-3.5 -115 +,-3.5 -108 +,-3.5 -107 +,-3.5 -112 +,-3.5 -110 +,-3.5 -105 +,-3.5 -110 +,-3.5 -108
9,460 Colts,+3.5 -115 +,+3.5 -105 +,+3.5 -112 +,+3.5 -114 +,+3.5 -108 +,+3.5 -115 +,+3.5 -115 +,+3.5 -108 +,+3.5 -112


In [11]:
cleaned = pd.DataFrame(columns=intermediate.columns)
for row in intermediate.iterrows():
    kill = False
    odds = []
    skip = False
    for source in sources:
        check = type(row[1][source]) == str
        skip += check
        odds.append(row[1][source])
    skip = not bool(skip)
    if skip:
        continue
    skip = True
    for odd in odds:
        if type(odd) != str:
            continue
        if odd[0] == 'o':
            kill = True
    if kill:
        break
    cleaned.loc[len(cleaned)] = row[1]
    
# cleaned = cleaned.drop('Unnamed: 3', axis=1)
# cleaned = cleaned.drop('Unnamed: 4', axis=1)
cleaned = cleaned.drop('Unnamed: 11', axis=1)
cleaned = cleaned.drop('Open', axis=1)
    
for i, row in cleaned.iterrows():
    for source in sources:
        if type(row[source]) != str:
            continue
        if row[source].split()[0][0] == '+':
            num = row[source].split()[0][1:]
        else:
            num = row[source].split()[0]
        if num == 'PK':
            num = 0.
        else:
            num = float(num)
        cleaned.loc[i, source] = float(num)
        
for i, row in cleaned.iterrows():
    result = ''.join([i for i in row['Time'] if not i.isdigit()])
    result = result[1:]
    cleaned.loc[i, 'Time'] = result
cleaned = cleaned.rename(columns={'Time': 'Team'})
cleaned['index'] = np.arange(len(cleaned))

ValueError: could not convert string to float: '--4.5'

In [12]:
# cleaned['ave_spread'] = cleaned.iloc[:, [1,2,3,4,5,6,7,8,9,10]].mean(axis=1)
cleaned['ave_spread'] = cleaned[sources].mean(axis=1)

TypeError: unsupported operand type(s) for +: 'float' and 'str'

In [ ]:
cleaned.sort_values(by='ave_spread')